# Twisted Tail Method

This page explains the numerical method behind the GWLENS wave-optics solvers, following

> S. Harikumar, A. Kanu, A. Nemmani, P. Ciecieląg, M. Bejger,
> *A twist in the tail: GWLENS, a fast and accurate tool for wave-optics lensing of gravitational waves*,
> [arXiv:2611.24621](https://arxiv.org/abs/2611.24621) (2026).

The method  **split the diffraction integral at a point beyond all the lensed images, integrate the
bounded *head* on the real axis with Gauss–Legendre quadrature, and rotate the semi-infinite *tail* into the complex
plane, where it decays exponentially and is integrated with Gauss–Laguerre quadrature.** Because the image region is
never deformed, the method never meets the poles and branch cuts that complicate full contour-deformation approaches.
For the point-mass lens it reproduces the exact solution to about $10^{-13}$.

**On this page**

1. Wave-optics lensing and the amplification factor
2. Why the integral is hard
3. Splitting the integral: head and tail
4. The head: composite Gauss–Legendre quadrature
5. The twist: rotating the tail
6. Many frequencies and many source positions
7. Non-axisymmetric lenses
8. A minimal implementation, checked against the exact point mass
9. The same calculation with `Solver1D`

## 1. Wave-optics lensing and the amplification factor

In the thin-lens approximation a gravitational wave $\tilde h(f)$ emitted by a source behind a lens reaches the
observer as

$$
\tilde h_{\rm L}(f) = F(f)\,\tilde h(f),
$$

where $F$ is the frequency-dependent **amplification factor**. Positions in the lens plane are measured in units of a
characteristic length $\xi_0$ (the Einstein radius for a point mass), giving the dimensionless lens-plane position
$\boldsymbol{x}$ and source position $\boldsymbol{y}$. The frequency enters through the dimensionless combination

$$
w = \frac{(1+z_{\rm L})\,D_{\rm S}\,\xi_0^2}{c\,D_{\rm L}\,D_{\rm LS}}\;2\pi f ,
\qquad\text{which for a point mass } M_{\rm L} \text{ is}\qquad
w = \frac{8\pi G M_{\rm L}(1+z_{\rm L})}{c^3}\,f .
$$

The amplification factor is the Fresnel–Kirchhoff diffraction integral over the lens plane,

$$
F(w,\boldsymbol{y}) = \frac{w}{2\pi i}\int_{\mathbb{R}^2} d^2x\;
\exp\!\big[\,i w\,T(\boldsymbol{x},\boldsymbol{y})\big],
\qquad
T(\boldsymbol{x},\boldsymbol{y}) = \tfrac12\,|\boldsymbol{x}-\boldsymbol{y}|^2 - \psi(\boldsymbol{x}) + \phi_m(\boldsymbol{y}),
$$

with $T$ the Fermat potential (dimensionless time delay), $\psi$ the lensing potential and $\phi_m(\boldsymbol y)$ a
constant chosen so that the first image arrives at $T=0$. Without a lens ($\psi=0$) the integral gives $F=1$.

**Geometric-optics limit.** When $w \gg 1$ the integral is dominated by its stationary points
$\nabla_{\boldsymbol x}T=0$, which are the images $\boldsymbol x_j$, and

$$
F_{\rm geo}(w,\boldsymbol y) = \sum_j |\mu_j|^{1/2}\,\exp\!\big(i w T_j - i\pi n_j\big),
\qquad n_j = 0,\ \tfrac12,\ 1 \ \text{for minima, saddles, maxima}.
$$

When the wavelength is comparable to the Schwarzschild radius of the lens ($w \lesssim 1$–$10$) diffraction matters
and the full integral has to be evaluated.

**Axisymmetric lenses.** If $\psi=\psi(|\boldsymbol x|)$, the angular integral gives a Bessel function, and with
$x = |\boldsymbol x|$, $y=|\boldsymbol y|$:

$$
F(w,y) = -i\,w\,e^{\,i w y^2/2}\int_0^\infty x\,J_0(w y x)\;
\exp\!\Big\{ i w\Big[\tfrac12 x^2 - \psi(x)\Big]\Big\}\,dx ,
$$

up to the constant phase $e^{iw\phi_m(y)}$. This one-dimensional integral is what `Solver1D` evaluates.

## 2. Why the integral is hard

Closed forms exist only for a handful of lenses (point mass, and series solutions for the singular isothermal
sphere). For anything else the integral must be computed numerically, and it is awkward:

* **It does not converge absolutely.** For large $x$, $|J_0(wyx)| \sim (wyx)^{-1/2}$ while the measure grows like
  $x$, so the modulus of the integrand *grows* like $x^{1/2}$ (and like $x$ when $y=0$). The integral exists only
  because of the cancellations of the phase $e^{iwx^2/2}$.
* **It oscillates fast.** The number of oscillations up to radius $x$ is about $w x^2/(4\pi)$, so at large $w$ a
  real-axis quadrature needs very many nodes.
* **The lens may be singular.** For the point mass $\psi=\ln x$, the phase $-w\ln x$ oscillates infinitely often as
  $x\to 0$.

Common remedies are regularising the integrand with a damping factor and extrapolating, subtracting the
geometric-optics asymptotics, or deforming the whole contour into the complex plane (for example along
Picard–Lefschetz thimbles). Full contour deformation works well but has to steer around the poles and branch
cuts of $\psi$ in the complex plane, and those live near the images. The twisted tail method deforms only the part
of the contour far from the images.

### The variable $u = x^2/2$

Substituting $u = x^2/2$ (so $x\,dx = du$) removes the explicit $x$ and makes the free phase linear:

$$
F(w,y) = -i\,w\,e^{\,i w y^2/2}\int_0^\infty J_0\!\big(w y \sqrt{2u}\big)\,e^{\,i w A(u)}\,du,
\qquad A(u) = u - \psi\!\big(\sqrt{2u}\big).
$$

Far from the lens $A(u)\approx u$, so the integrand behaves like $e^{iwu}$: a pure Fourier oscillation in $u$.

## 3. Splitting the integral: head and tail

Choose a split point $u_b = x_b^2/2$ and write

$$
\int_0^\infty (\dots)\,du \;=\;
\underbrace{\int_0^{u_b}(\dots)\,du}_{\text{head } H(w,y)}
\;+\;
\underbrace{\int_{u_b}^\infty(\dots)\,du}_{\text{tail } \mathcal T(w,y)} ,
\qquad
F = -i\,w\,e^{\,iwy^2/2}\,\big[H + \mathcal T\big].
$$

The split is placed beyond every image. Images on the lens axis satisfy the lens equation
$x \mp y = \psi'(x)$. GWLENS picks the smallest $x_b \ge \max(3,\,2y_{\max})$ (growing it by 10 % at a time) such that

$$
\frac{y_{\max} + \psi'(x_b)}{x_b} \;\le\; \frac12 ,
$$

where $y_{\max}$ is the largest source position the solver must serve. Beyond such a point the free term $x^2/2$
dominates both the source offset and the lens deflection, so there are no stationary points and the integrand is
"almost free". As shown below, the same quantity sets the decay rate of the rotated tail.

If the potential contains a mass sheet, $\psi'(x)/x \to q$ as $x\to\infty$, and the threshold becomes $(1+q)/2$.
The construction requires $q < 1$; lenses with $q\ge 1$ (Type II and III images) cannot have their tail rotated upward
and are not handled by this method.

At low frequencies the head is also lengthened, $u_b = \max\!\big(x_b^2/2,\ 20/w_{\min}\big)$ for a band whose lowest
frequency is $w_{\min}$.

## 4. The head: composite Gauss–Legendre quadrature

The head $[0,u_b]$ is a bounded interval that contains all the images and all the physics of the lens. It is
integrated on the real axis, so nothing about the analytic structure of $\psi$ matters here.

The interval is cut into pieces $[e_p, e_{p+1}]$, and an $n_p$-point Gauss–Legendre rule with nodes $t_k$ and
weights $\omega_k$ is used on each:

$$
H(w,y) \approx \sum_{j} W_j\,J_0(w y x_j)\,e^{\,i w A_j},
\qquad
u_j = m_p + h_p t_k,\quad W_j = h_p\,\omega_k,\quad x_j=\sqrt{2u_j},\quad A_j = u_j - \psi(x_j),
$$

with $m_p$ and $h_p$ the midpoint and half-width of piece $p$. The pieces are chosen so that:

* each piece holds at most `cycles` (default 8) phase cycles at the highest frequency of the band, measured with a
  running "phase meter" $V(u) = \int |dA| + y_{\max}\,x$ that also counts the oscillation of $J_0$;
* the edges are graded geometrically towards $u=0$, where $x=\sqrt{2u}$ and many potentials (e.g. $\ln x$) are
  singular;
* with `adaptive_gl=True`, pieces with little phase get fewer than `ngl` (default 32) nodes, sized by a
  Bernstein-ellipse error bound for $e^{i\kappa t}g(t)$ on $[-1,1]$ with $g$ singular at the nearest singular point.

**The key property for speed:** the nodes $u_j$, weights $W_j$ and phases $A_j$ do not depend on $w$ or $y$.
$\psi$ is evaluated once when the solver is built. For a fixed $y$, $H(w,y)$ is a non-uniform Fourier sum in $w$, so
many frequencies at once are a type-3 non-uniform FFT (FINUFFT). Nodes near $u=0$, whose phase barely changes across
the band, are summed once into a Taylor polynomial in $w$.

## 5. The twist: rotating the tail

### Cauchy's theorem

On $[u_b,\infty)$ the integrand never decays. But if $\psi$ is analytic there, the integrand

$$
g(u) = J_0\!\big(w y\sqrt{2u}\big)\,e^{\,iw[u - \psi(\sqrt{2u})]}
$$

is analytic in the quarter plane $\{\,\mathrm{Re}\,u > u_b,\ \mathrm{Im}\,u>0\,\}$. Close the contour with the
vertical ray $u = u_b + i t$, $t\ge 0$, and an arc at infinity. Because $e^{iwu}$ decays in the upper half plane, the
arc contributes nothing (Jordan's lemma), and Cauchy's theorem gives

$$
\mathcal T(w,y) = \int_{u_b}^{\infty} g(u)\,du = i\int_0^{\infty} g(u_b + i t)\,dt .
$$

Only the tail is rotated, and it is rotated about a point beyond the images. The head, where $\psi$ may have its
singularities and where the stationary points sit, is never deformed. **This is the "twist in the tail".**

### How fast does the rotated tail decay?

On the vertical ray, $x(t) = \sqrt{2(u_b + it)}$. Since $\mathrm{Re}\,x \ge x_b$,

$$
\mathrm{Im}\,x(t) = \frac{t}{\mathrm{Re}\,x(t)} \;\le\; \frac{t}{x_b}.
$$

The three factors of $g$ then behave as

$$
\big|e^{iwu}\big| = e^{-wt},\qquad
\big|J_0(wyx)\big| \lesssim e^{\,w y\,\mathrm{Im}\,x} \le e^{\,w y t / x_b},\qquad
\big|e^{-iw\psi(x)}\big| \approx e^{\,w\,\psi'(x_b)\,\mathrm{Im}\,x} \le e^{\,w\,\psi'(x_b)\,t/x_b},
$$

so the integrand decays like $e^{-\alpha w t}$ with

$$
\alpha = 1 - \frac{y_{\max} + \psi'(x_b)}{x_b} \;\ge\; \frac12 .
$$

This is exactly why the split criterion of Section 3 is the right one: it guarantees a decay rate of at least
$w/2$ on the rotated tail.

### Gauss–Laguerre quadrature

Substituting $t = s/(\alpha w)$ turns the decay into $e^{-s}$, the weight function of Gauss–Laguerre quadrature.
Factoring out the value of the phase at the split point,

$$
\mathcal T(w,y) = \frac{i}{\alpha w}\,e^{\,iw[u_b - \psi(x_b)]}
\int_0^\infty e^{-s}\;\Big\{ e^{\,s(1 - 1/\alpha)}\;
e^{-iw[\psi(x(s)) - \psi(x_b)]}\;J_0\big(w y\,x(s)\big)\Big\}\,ds ,
\qquad x(s) = \sqrt{2\Big(u_b + \frac{i s}{\alpha w}\Big)} .
$$

The braces hold a smooth, slowly varying function, so an $N$-point Gauss–Laguerre rule with nodes $s_k$ and weights
$\lambda_k$ (default `nlag` = 30) converges exponentially:

$$
\boxed{\;
\mathcal T(w,y) \approx \frac{i}{\alpha w}\,e^{\,iw[u_b - \psi(x_b)]}
\sum_{k=1}^{N} \lambda_k\,e^{\,s_k(1-1/\alpha)}\,
e^{-iw[\psi(x_k)-\psi(x_b)]}\,J_0(w y\,x_k)\;}
\qquad x_k = \sqrt{2\big(u_b + i s_k/(\alpha w)\big)} .
$$

The tail costs about 30 evaluations of $\psi$ and $J_0$ at complex arguments per frequency, whatever the value of $w$.

### Large arguments: splitting $J_0$ into Hankel functions

When $w y x_b$ is large, $J_0$ itself oscillates. Writing
$J_0(z) = \tfrac12\big[H_0^{(1)}(z) + H_0^{(2)}(z)\big]$ and pulling out $e^{\pm i z_b}$ with $z_b = wyx_b$,

$$
J_0(w y x_k) = \tfrac12\Big[e^{\,i z_b}\,h_1(w y x_k)\,e^{\,iwy(x_k - x_b)}
+ e^{-i z_b}\,h_2(w y x_k)\,e^{-iwy(x_k - x_b)}\Big],
\qquad h_{1,2}(z) = H_0^{(1,2)}(z)\,e^{\mp i z},
$$

leaves two smooth envelopes $h_{1,2}$. These are computed with ascending series for small $|z|$ and the Hankel
asymptotic series (DLMF 10.17.5) for large $|z|$, which avoids slow general-purpose complex Bessel routines.

### The one rule: $\psi$ must be analytic

The rotation is valid only if $\psi$, written as a function of $x$, extends analytically into the region swept by
the tail. In practice the potential must be written with analytic operations only (`np.log`, `np.sqrt`,
`np.arctanh`, polynomials …) and must accept complex arrays. Functions such as `abs`, `np.where`, `np.real` or
`if/else` on $x$ break analyticity and make the rotated tail wrong.

GWLENS checks this when a solver is built (`Potential.check_analytic`). It compares the complex-step derivative

$$
\psi'(x) = \frac{\mathrm{Im}\,\psi(x + i h)}{h} + \mathcal O(h^2), \qquad h = 10^{-20},
$$

which is exact to rounding only for an analytic $\psi$, with a central finite difference, and checks that
$\mathrm{Im}\,\psi = 0$ on the real axis. The same complex-step derivative gives $\psi'(x_b)$ for the split criterion
and for $\alpha$.

## 6. Many frequencies and many source positions

Parameter estimation needs $F$ on thousands of frequencies, often for many source positions. The structure of the
method lets almost everything be precomputed.

**Frequency bands.** $[w_{\min}, w_{\max}]$ is divided into geometric bands $[w_{\rm hi}/r,\ w_{\rm hi}]$
(`band_ratio` $r$ = 8 by default). Each band has its own head nodes (sized for its $w_{\rm hi}$) and its own split
point. Bands are built lazily, only when a frequency in them is requested.

**Tail: Chebyshev interpolation in $\ln w$.** Inside a band the tail sum is a smooth function of $w$ once the factor
$e^{iw[u_b-\psi(x_b)]}$ (and $e^{\pm i w y x_b}$ in the Hankel form) is removed. It is computed at `n_cheb`
Chebyshev points in $\ln w$ and evaluated anywhere in the band by barycentric interpolation. The tail weights
$\lambda_k e^{s_k(1-1/\alpha)}e^{-iw[\psi(x_k)-\psi(x_b)]}$ do not depend on $y$, so they are reused for every source
position.

**Head: non-uniform FFT.** For one source position and many frequencies, the head sum
$\sum_j W_j J_0(wyx_j)\,e^{iwA_j}$ is evaluated with a type-3 NUFFT. Where $wyx_j$ is large, $J_0$ is again split into
Hankel envelopes times $e^{\pm iwyx_j}$, so the sum stays a Fourier sum with shifted nodes $A_j \pm y x_j$.

**Source positions.** Building a solver with `y_radius` > 0 sizes the split point for $y_{\max} = y + y_{\rm radius}$.
The same nodes then serve every $y$ in the range (`profile`, `evaluate`), which is where the method is most efficient
compared with methods that rebuild their contour for every source position.

**Interpolation in $w$ for the full $F$.** With `interpolate=True`, $F$ itself is computed on Chebyshev panels sized
by the image time delays and interpolated to the requested frequencies. Each panel is checked against exact values
and refined if needed.

**Time origin.** By default (`time_origin="first_image"`) GWLENS multiplies $F$ by $e^{-iwT_{\min}(y)}$, where
$T_{\min}$ is the global minimum of the Fermat potential, i.e. $\phi_m(y)$ of the point-mass solution. This matches
the phase convention of other codes. `first_image_delay(y)` returns $T_{\min}(y)$.

With the default settings the result is accurate to about $10^{-12}$. The lighter `PE_SETTINGS` give about $10^{-8}$ and
are meant for parameter estimation.

## 7. Non-axisymmetric lenses

For a general $\psi(\boldsymbol x)$ (binary lenses, external shear, offset subhaloes), `Solver2D` applies the same
idea in polar coordinates $\boldsymbol x = \boldsymbol c + r(\cos\theta,\sin\theta)$ about a centre $\boldsymbol c$:

$$
F(w,\boldsymbol y) = \frac{w}{2\pi i}\int_0^{2\pi}\! d\theta \int_0^\infty r\,dr\;
\exp\!\Big\{ i w\Big[\tfrac12|\boldsymbol c + r\hat{\boldsymbol n}_\theta - \boldsymbol y|^2
- \psi(\boldsymbol c + r\hat{\boldsymbol n}_\theta)\Big]\Big\}.
$$

* The **head**, a disc of radius $x_b$ around $\boldsymbol c$ that contains all images, is a real 2D grid of Gauss–Legendre
  nodes in $u=r^2/2$ with patches around the singular points of the lens. It is evaluated with a 1D NUFFT for a
  spectrum, or a 2D NUFFT for a map over source positions.
* The **tail** is rotated along every direction: $u = u_b + it$ for each $\theta$, with Gauss–Laguerre nodes in $t$.
  The number of directions is set by the angular bandwidth of the integrand along the rotated contour.
* The decay condition becomes a condition on the large-$r$ behaviour of the lens: with convergence $\kappa$ and shear
  $\gamma$ at infinity, the tail can be rotated upward in every direction only if $\kappa + |\gamma| < 1$.
* The centre $\boldsymbol c$ is chosen automatically (`center="auto"`) among the singular points and their centroid
  by estimating the grid size of each choice.

The analyticity rule applies unchanged: $\psi(x_1,x_2)$ must accept complex arrays and use analytic operations only.